In [ ]:
"""
NOTEBOOK 5 - BUSCA DE MODELO: VERSÃO EXPLORATÓRIA (A) + VERSÃO VALIDADA (B)
=============================================================================
Eye tracking - TEA vs Controle - 50 faces (30 humanas: feliz/neutra/raiva,
20 desenhos: neutro/raiva/feliz)

LEIA ISTO ANTES DE USAR OS NÚMEROS DESTE SCRIPT:

Este arquivo tem DUAS seções que respondem à mesma pergunta ("qual
combinação de condição emocional + features + modelo dá a melhor
acurácia?") de duas formas diferentes:

  PARTE A - BUSCA EXPLORATÓRIA (ampla, sem correção estatística)
     -> testa TODAS as condições (raiva humano, raiva humano+desenho,
        feliz humano, feliz humano+desenho, etc.), várias contagens de
        features (K) e várias seeds.
     -> É EXATAMENTE o tipo de varredura que você já vinha fazendo nos
        notebooks anteriores.
     -> PROBLEMA: com ~38 pacientes, testar dezenas de combinações e
        reportar "a melhor" é igual a jogar uma moeda 50 vezes e
        anunciar a sequência mais bonita. O número que sair daqui NÃO
        pode ser citado como acurácia do seu classificador. Serve só
        para gerar HIPÓTESES (ex: "raiva parece mais informativa que
        felicidade") que serão testadas de verdade na Parte B.
     -> Cada print desta seção vem com o aviso [EXPLORATÓRIO - NÃO
        VALIDADO] para você nunca confundir com o resultado final.

  PARTE B - VERSÃO VALIDADA (nested CV + teste de permutação + FDR)
     -> Pega os cenários candidatos (definidos por você, ou os
        melhores da Parte A) e avalia com:
          1. Seleção de K de features feita DENTRO de cada fold de
             treino (sem vazamento) via CV interno.
          2. Para modelos estocásticos (RF, XGBoost): média ± desvio
             padrão sobre várias seeds, nunca o valor máximo.
          3. Teste de permutação (embaralha os rótulos N vezes e
             recalcula o pipeline inteiro) -> p-valor empírico real:
             qual a chance de obter essa acurácia com rótulos
             aleatórios?
          4. Correção de Benjamini-Hochberg (FDR) pelo número total de
             cenários testados na Parte A, porque "testei 15 coisas e
             uma deu p=0.04" não é a mesma coisa que "testei 1 coisa e
             deu p=0.04".
     -> O número que sai daqui é o que pode, de fato, ser reportado.

Ajuste CAMINHO_MATRIZ e PASTA_CSV para o seu ambiente antes de rodar.
"""

import numpy as np
import pandas as pd
import warnings
from itertools import product

from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

warnings.filterwarnings("ignore")

CAMINHO_MATRIZ = "/workspaces/EyeTracking/data/processed/matriz_features_ml.csv"

RANDOM_SEEDS = list(range(20))   # seeds usadas para MÉDIA (nunca para escolher a melhor)
K_GRID = [4, 6, 8, 10, 12, 15]   # candidatos de nº de features, testados sem vazamento na Parte B
N_PERMUTATIONS = 500             # para o teste de permutação da Parte B (nos melhores cenários)
TOP_N_PARA_VALIDAR = 5           # quantos cenários da Parte A vão para o teste de permutação da Parte B

# ---------------------------------------------------------------------------
# CARREGAMENTO
# ---------------------------------------------------------------------------

df = pd.read_csv(CAMINHO_MATRIZ)
y = (df["Grupo"] == "TEA").astype(int)
X_total = df.drop(columns=["Paciente", "Grupo"])
if "Total_Linhas_Face" in X_total.columns:
    X_total = X_total.drop(columns=["Total_Linhas_Face"])

print(f"Base: {X_total.shape[0]} pacientes | {X_total.shape[1]} features | "
      f"{sum(y==1)} TEA / {sum(y==0)} Controle")

VARS_GLOBAIS = [c for c in [
    "Velocidade_Sacadica_Media", "Num_Total_Fixacoes", "Duracao_Media_Fixacao_ms",
    "Area_Dispersao_ConvexHull", "Entropia_Shannon_Olhar", "Num_Transicoes_Olho_Boca",
    "Pupila_Baseline", "Pupila_Face_Global", "Pupila_Reatividade_Global",
    "Fuga_Visual_Global_%", "EMI_Global", "Indice_Social_Global", "TTFF_Medio_Olhos_ms",
] if c in X_total.columns]


def colunas_por(tipo=None, emocao=None):
    cols = X_total.columns
    if tipo:
        cols = [c for c in cols if tipo in c]
    if emocao:
        cols = [c for c in cols if emocao in c]
    return list(cols)


CENARIOS = {
    "Globais apenas":            list(set(VARS_GLOBAIS)),
    "Humano Raiva":               list(set(colunas_por("Humano", "Raiva") + VARS_GLOBAIS)),
    "Humano Feliz":                list(set(colunas_por("Humano", "Feliz") + VARS_GLOBAIS)),
    "Humano Neutro":               list(set(colunas_por("Humano", "Neutro") + VARS_GLOBAIS)),
    "Desenho Raiva":               list(set(colunas_por("Desenho", "Raiva") + VARS_GLOBAIS)),
    "Desenho Feliz":                list(set(colunas_por("Desenho", "Feliz") + VARS_GLOBAIS)),
    "Desenho Neutro":               list(set(colunas_por("Desenho", "Neutro") + VARS_GLOBAIS)),
    "Raiva (Humano+Desenho)":       list(set(colunas_por(emocao="Raiva") + VARS_GLOBAIS)),
    "Feliz (Humano+Desenho)":       list(set(colunas_por(emocao="Feliz") + VARS_GLOBAIS)),
    "Neutro (Humano+Desenho)":      list(set(colunas_por(emocao="Neutro") + VARS_GLOBAIS)),
    "Tudo (57 features)":           list(X_total.columns),
}
CENARIOS = {k: v for k, v in CENARIOS.items() if len(v) > 0}

MODELOS_FIXOS = {
    "SVM Linear":    lambda seed: SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=seed),
    "Reg. Logistica": lambda seed: LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed),
    "KNN":            lambda seed: KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "Random Forest":  lambda seed: RandomForestClassifier(n_estimators=200, max_depth=3, class_weight="balanced", random_state=seed),
    "XGBoost":        lambda seed: xgb.XGBClassifier(max_depth=3, learning_rate=0.05, n_estimators=150,
                                                       scale_pos_weight=(sum(y == 0) / sum(y == 1)),
                                                       eval_metric="logloss", random_state=seed),
}

MODELOS_ESTOCASTICOS = {"KNN": False, "SVM Linear": False, "Reg. Logistica": True,
                         "Random Forest": True, "XGBoost": True}


def metrics_from_preds(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    acc = (y_true == y_pred).mean()
    vp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    vn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    sens = vp / (vp + fn) * 100 if (vp + fn) > 0 else 0
    espec = vn / (vn + fp) * 100 if (vn + fp) > 0 else 0
    return acc * 100, sens, espec


# =============================================================================
# PARTE A - BUSCA EXPLORATÓRIA (não validada — só gera hipóteses)
# =============================================================================
print("\n" + "=" * 90)
print("PARTE A - BUSCA EXPLORATÓRIA [EXPLORATÓRIO - NÃO VALIDADO]")
print("Testa todos os cenários x modelos x K com LOOCV simples (K fixo por fora do fold).")
print("=" * 90)

resultados_A = []

for nome_cenario, colunas in CENARIOS.items():
    X_cenario = X_total[colunas]
    max_k = min(max(K_GRID), X_cenario.shape[1])
    k_candidatos = [k for k in K_GRID if k <= max_k] or [max_k]

    for nome_modelo, construtor in MODELOS_FIXOS.items():
        estocastico = MODELOS_ESTOCASTICOS[nome_modelo]
        seeds_usadas = RANDOM_SEEDS if estocastico else [0]

        for k in k_candidatos:
            accs_por_seed = []
            sens_por_seed = []
            espec_por_seed = []

            for seed in seeds_usadas:
                modelo = construtor(seed)
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("select", SelectKBest(f_classif, k=k)),
                    ("clf", modelo),
                ])
                loo = LeaveOneOut()
                preds, trues = [], []
                for tr_idx, te_idx in loo.split(X_cenario):
                    pipe.fit(X_cenario.iloc[tr_idx], y.iloc[tr_idx])
                    preds.append(pipe.predict(X_cenario.iloc[te_idx])[0])
                    trues.append(y.iloc[te_idx].values[0])
                acc, sens, espec = metrics_from_preds(trues, preds)
                accs_por_seed.append(acc)
                sens_por_seed.append(sens)
                espec_por_seed.append(espec)

            resultados_A.append({
                "Cenario": nome_cenario,
                "Modelo": nome_modelo,
                "K": k,
                "N_Features_Disponiveis": len(colunas),
                "Acc_Media_%": np.mean(accs_por_seed),
                "Acc_DP_%": np.std(accs_por_seed),
                "Sens_Media_%": np.mean(sens_por_seed),
                "Espec_Media_%": np.mean(espec_por_seed),
                "N_seeds": len(seeds_usadas),
            })

df_A = pd.DataFrame(resultados_A).sort_values("Acc_Media_%", ascending=False)
n_testes_totais = len(df_A)

print(f"\n[EXPLORATÓRIO - NÃO VALIDADO] Total de combinações testadas: {n_testes_totais}")
print("[EXPLORATÓRIO - NÃO VALIDADO] Top 10 por acurácia média (K fixo, sem correção):\n")
print(df_A.head(10).to_string(index=False))
print("\n[AVISO] Estes números tendem a estar inflados por acaso amostral.")
print("        Note que a acurácia é a MÉDIA sobre seeds (não o melhor caso) — mesmo assim,")
print(f"        testamos {n_testes_totais} combinações, então o 'melhor' daqui ainda precisa")
print("        passar pelo teste de permutação da Parte B antes de significar algo.")



Base: 38 pacientes | 62 features | 17 TEA / 21 Controle

PARTE A - BUSCA EXPLORATÓRIA [EXPLORATÓRIO - NÃO VALIDADO]
Testa todos os cenários x modelos x K com LOOCV simples (K fixo por fora do fold).


In [1]:
import numpy as np
import pandas as pd
import warnings
from itertools import product

from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

warnings.filterwarnings("ignore")

CAMINHO_MATRIZ = "/workspaces/EyeTracking/data/processed/matriz_features_ml.csv"

RANDOM_SEEDS = list(range(20))   # seeds usadas para MÉDIA (nunca para escolher a melhor)
K_GRID = [4, 6, 8, 10, 12, 15]   # candidatos de nº de features, testados sem vazamento na Parte B
N_PERMUTATIONS = 500             # para o teste de permutação da Parte B (nos melhores cenários)
TOP_N_PARA_VALIDAR = 5           # quantos cenários da Parte A vão para o teste de permutação da Parte B

# ---------------------------------------------------------------------------
# CARREGAMENTO
# ---------------------------------------------------------------------------

df = pd.read_csv(CAMINHO_MATRIZ)
y = (df["Grupo"] == "TEA").astype(int)
X_total = df.drop(columns=["Paciente", "Grupo"])
if "Total_Linhas_Face" in X_total.columns:
    X_total = X_total.drop(columns=["Total_Linhas_Face"])

print(f"Base: {X_total.shape[0]} pacientes | {X_total.shape[1]} features | "
      f"{sum(y==1)} TEA / {sum(y==0)} Controle")

VARS_GLOBAIS = [c for c in [
    "Velocidade_Sacadica_Media", "Num_Total_Fixacoes", "Duracao_Media_Fixacao_ms",
    "Area_Dispersao_ConvexHull", "Entropia_Shannon_Olhar", "Num_Transicoes_Olho_Boca",
    "Pupila_Baseline", "Pupila_Face_Global", "Pupila_Reatividade_Global",
    "Fuga_Visual_Global_%", "EMI_Global", "Indice_Social_Global", "TTFF_Medio_Olhos_ms",
] if c in X_total.columns]


def colunas_por(tipo=None, emocao=None):
    cols = X_total.columns
    if tipo:
        cols = [c for c in cols if tipo in c]
    if emocao:
        cols = [c for c in cols if emocao in c]
    return list(cols)


CENARIOS = {
    "Globais apenas":            list(set(VARS_GLOBAIS)),
    "Humano Raiva":               list(set(colunas_por("Humano", "Raiva") + VARS_GLOBAIS)),
    "Humano Feliz":                list(set(colunas_por("Humano", "Feliz") + VARS_GLOBAIS)),
    "Humano Neutro":               list(set(colunas_por("Humano", "Neutro") + VARS_GLOBAIS)),
    "Desenho Raiva":               list(set(colunas_por("Desenho", "Raiva") + VARS_GLOBAIS)),
    "Desenho Feliz":                list(set(colunas_por("Desenho", "Feliz") + VARS_GLOBAIS)),
    "Desenho Neutro":               list(set(colunas_por("Desenho", "Neutro") + VARS_GLOBAIS)),
    "Raiva (Humano+Desenho)":       list(set(colunas_por(emocao="Raiva") + VARS_GLOBAIS)),
    "Feliz (Humano+Desenho)":       list(set(colunas_por(emocao="Feliz") + VARS_GLOBAIS)),
    "Neutro (Humano+Desenho)":      list(set(colunas_por(emocao="Neutro") + VARS_GLOBAIS)),
    "Tudo (57 features)":           list(X_total.columns),
}
CENARIOS = {k: v for k, v in CENARIOS.items() if len(v) > 0}

MODELOS_FIXOS = {
    "SVM Linear":    lambda seed: SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=seed),
    "Reg. Logistica": lambda seed: LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed),
    "KNN":            lambda seed: KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "Random Forest":  lambda seed: RandomForestClassifier(n_estimators=200, max_depth=3, class_weight="balanced", random_state=seed),
    "XGBoost":        lambda seed: xgb.XGBClassifier(max_depth=3, learning_rate=0.05, n_estimators=150,
                                                       scale_pos_weight=(sum(y == 0) / sum(y == 1)),
                                                       eval_metric="logloss", random_state=seed),
}

MODELOS_ESTOCASTICOS = {"KNN": False, "SVM Linear": False, "Reg. Logistica": True,
                         "Random Forest": True, "XGBoost": True}


def metrics_from_preds(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    acc = (y_true == y_pred).mean()
    vp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    vn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    sens = vp / (vp + fn) * 100 if (vp + fn) > 0 else 0
    espec = vn / (vn + fp) * 100 if (vn + fp) > 0 else 0
    return acc * 100, sens, espec


# =============================================================================
# PARTE B - VERSÃO VALIDADA (nested CV + permutação + FDR)
# =============================================================================
print("\n" + "=" * 90)
print("PARTE B - VALIDAÇÃO RIGOROSA DOS MELHORES CANDIDATOS DA PARTE A")
print("=" * 90)

top_candidatos = (
    df_A.sort_values("Acc_Media_%", ascending=False)
    .drop_duplicates(subset=["Cenario", "Modelo"])
    .head(TOP_N_PARA_VALIDAR)
)


def roda_nested_loocv(X_cenario, nome_modelo, construtor, seed, k_grid):
    """LOOCV onde o K de SelectKBest é escolhido DENTRO de cada fold de treino
    (via um CV interno de 5 folds), evitando vazamento de informação do
    paciente de teste na escolha do número de features."""
    loo = LeaveOneOut()
    preds, trues = [], []

    for tr_idx, te_idx in loo.split(X_cenario):
        X_tr, X_te = X_cenario.iloc[tr_idx], X_cenario.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        # --- escolha do K só dentro do treino (CV interno) ---
        melhor_k, melhor_acc_interna = k_grid[0], -1
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for k in k_grid:
            k_real = min(k, X_tr.shape[1])
            inner_preds, inner_trues = [], []
            for itr, ite in inner_cv.split(X_tr, y_tr):
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("select", SelectKBest(f_classif, k=k_real)),
                    ("clf", construtor(seed)),
                ])
                pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
                inner_preds.extend(pipe.predict(X_tr.iloc[ite]))
                inner_trues.extend(y_tr.iloc[ite])
            acc_interna = np.mean(np.array(inner_preds) == np.array(inner_trues))
            if acc_interna > melhor_acc_interna:
                melhor_acc_interna, melhor_k = acc_interna, k_real

        # --- treino final do fold com o K escolhido internamente ---
        pipe_final = Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=melhor_k)),
            ("clf", construtor(seed)),
        ])
        pipe_final.fit(X_tr, y_tr)
        preds.append(pipe_final.predict(X_te)[0])
        trues.append(y_te.values[0])

    return np.array(trues), np.array(preds)


resultados_B = []

for _, linha in top_candidatos.iterrows():
    nome_cenario, nome_modelo = linha["Cenario"], linha["Modelo"]
    X_cenario = X_total[CENARIOS[nome_cenario]]
    construtor = MODELOS_FIXOS[nome_modelo]
    estocastico = MODELOS_ESTOCASTICOS[nome_modelo]
    seeds_usadas = RANDOM_SEEDS[:5] if estocastico else [0]  # nested já é caro; poucas seeds, sempre em média

    print(f"\nValidando: {nome_cenario} / {nome_modelo} ...")

    accs_obs = []
    for seed in seeds_usadas:
        trues, preds = roda_nested_loocv(X_cenario, nome_modelo, construtor, seed, K_GRID)
        acc, sens, espec = metrics_from_preds(trues, preds)
        accs_obs.append((acc, sens, espec))

    acc_obs = np.mean([a for a, s, e in accs_obs])
    sens_obs = np.mean([s for a, s, e in accs_obs])
    espec_obs = np.mean([e for a, s, e in accs_obs])

    # --- teste de permutação: embaralha os rótulos e refaz TUDO ---
    seed_perm = seeds_usadas[0]
    contagem_maior_igual = 0
    for i in range(N_PERMUTATIONS):
        y_embaralhado = y.sample(frac=1, random_state=1000 + i).reset_index(drop=True)
        y_bak = y.copy()
        globals()["y"] = y_embaralhado  # roda_nested_loocv usa y global; troca temporária
        try:
            trues_p, preds_p = roda_nested_loocv(X_cenario, nome_modelo, construtor, seed_perm, K_GRID)
            acc_p, _, _ = metrics_from_preds(trues_p, preds_p)
        finally:
            globals()["y"] = y_bak
        if acc_p >= acc_obs:
            contagem_maior_igual += 1

    p_valor = (contagem_maior_igual + 1) / (N_PERMUTATIONS + 1)

    resultados_B.append({
        "Cenario": nome_cenario,
        "Modelo": nome_modelo,
        "Acc_Validada_%": acc_obs,
        "Sens_Validada_%": sens_obs,
        "Espec_Validada_%": espec_obs,
        "p_valor_permutacao": p_valor,
    })
    print(f"  Acc validada (nested, média de seeds): {acc_obs:.1f}%  |  "
          f"Sens: {sens_obs:.1f}%  |  Espec: {espec_obs:.1f}%  |  p (permutação) = {p_valor:.4f}")

df_B = pd.DataFrame(resultados_B).sort_values("Acc_Validada_%", ascending=False)

# --- correção de Benjamini-Hochberg pelo Nº TOTAL de cenários testados na Parte A ---
df_B = df_B.sort_values("p_valor_permutacao").reset_index(drop=True)
m = n_testes_totais  # número total de hipóteses "testadas" (Parte A inteira)
df_B["rank"] = np.arange(1, len(df_B) + 1)
df_B["p_ajustado_BH"] = np.minimum.accumulate(
    (df_B["p_valor_permutacao"] * m / df_B["rank"])[::-1]
)[::-1].clip(upper=1.0)

print("\n" + "=" * 90)
print("RESULTADO FINAL VALIDADO (o único que deve ser citado)")
print("=" * 90)
print(df_B[["Cenario", "Modelo", "Acc_Validada_%", "Sens_Validada_%",
            "Espec_Validada_%", "p_valor_permutacao", "p_ajustado_BH"]].to_string(index=False))

print(f"\nCorreção aplicada considerando {m} combinações testadas na Parte A.")
print("Um cenário só deve ser reportado como sinal real se p_ajustado_BH < 0.05")
print("(idealmente < 0.01 dado o número de comparações feitas).")

df_A.to_csv("/mnt/user-data/outputs/parteA_exploratorio_NAO_VALIDADO.csv", index=False)
df_B.to_csv("/mnt/user-data/outputs/parteB_resultado_validado.csv", index=False)
print("\nCSVs salvos: parteA_exploratorio_NAO_VALIDADO.csv | parteB_resultado_validado.csv")

KeyboardInterrupt: 